In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# Load your data
# df_external = pd.read_csv('external_temp.csv')
# df_internal = pd.read_csv('internal_sensors.csv')

# Ensure datetime format
df_external['date'] = pd.to_datetime(df_external['date'])
df_internal['date'] = pd.to_datetime(df_internal['date'])

# Merge external and internal data on date
df_merged = pd.merge(df_internal, df_external, on='date', how='left')

# Feature engineering
def create_features(df):
    df['dayofyear'] = df['date'].dt.dayofyear
    df['month'] = df['date'].dt.month
    df['dayofweek'] = df['date'].dt.dayofweek
    return df

df_external = create_features(df_external)
df_merged = create_features(df_merged)

# -------------------------------
# Stage 1: Pre-train on external temperature
# -------------------------------
X_ext = df_external[['dayofyear', 'month', 'dayofweek']]
y_ext = df_external['external_temp']

model_ext = lgb.LGBMRegressor(n_estimators=100)
model_ext.fit(X_ext, y_ext)

# -------------------------------
# Stage 2: Fine-tune on internal sensor data
# -------------------------------

# Create features for fine-tuning
X_int = df_merged[['dayofyear', 'month', 'dayofweek', 'external_temp', 'humidity']]
y_int = df_merged['internal_temp']

# You can freeze the external model and use its predictions as a feature
df_merged['external_pred'] = model_ext.predict(df_merged[['dayofyear', 'month', 'dayofweek']])
X_int = df_merged[['dayofyear', 'month', 'dayofweek', 'external_pred', 'humidity']]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_int, y_int, test_size=0.2, random_state=42)

# Train internal model
model_internal = lgb.LGBMRegressor(n_estimators=100)
model_internal.fit(X_train, y_train)

# Evaluation
y_pred = model_internal.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE on internal data: {rmse:.2f}')
